# Fine-tuning Protein Language Models with ESM2

This practical compares simple and parameter-efficient ways to adapt a pretrained protein language model to a binary peptide aggregation task.

### What you will learn
By the end of this notebook, you will be able to:

1. Load peptide train/test datasets and check whether the split is sensible.
2. Visualize learned ESM2 sequence representations with UMAP.
3. Compare a frozen ESM2 feature extractor, LoRA fine-tuning, full fine-tuning, and an amino-acid-count baseline.
4. Reason about validation/test splits, class balance, parameter efficiency, and statistical uncertainty.

### The four strategies in this notebook
We will compare:

- **Amino-acid-count baseline:** use 20 amino-acid composition features with logistic regression or random forest.
- **Frozen ESM2 baseline:** use ESM2 as a fixed feature extractor and train only a small classifier.
- **LoRA / PEFT:** inject a small number of trainable low-rank parameters into selected transformer modules.
- **Full fine-tuning:** update the whole ESM2 backbone and train end-to-end.

### Practical notes
- A GPU is strongly recommended, for example in Google Colab.
- We use ESM2-8M, the smallest ESM2 variant, so the workflow stays workshop-friendly.
- Expected files are explained in Section 3. In this version, the expected filenames are `train_data.csv` and `test_data.csv`.
- The code tasks are deliberately small and marked as **STUDENT TASK 1**, **STUDENT TASK 2**, ... . For each task, read the documentation link and answer the nearby numbered questions before filling in code.

---

## Table of contents
1. [Environment setup](#setup)
2. [Background: why fine-tune protein language models?](#background)
3. [Dataset and task](#data)
4. [Model architecture overview](#architecture)
5. [Amino-acid-count baseline](#aacount-baseline)
6. [Inspecting pretrained sequence embeddings with UMAP](#umap)
7. [Strategy 1: frozen baseline](#frozen)
8. [Strategy 2: LoRA fine-tuning](#lora)
9. [Strategy 3: full fine-tuning](#full)
10. [Comparison and interpretation](#comparison)
11. [Takeaways and exercises](#exercises)


## Exercise style

Work through the notebook and pay attention to every design choice. The goal is not only to make the code run, but to understand why the modeling choices matter.

Before running a section, first predict what it should do and why that choice makes sense for protein sequence modeling. During the practical, discuss your answers with a neighbour or ask for feedback.

---
## 1. Environment setup <a id='setup'></a>

We first install and import the packages required for the tutorial. We also set seeds and global parameters.

Package notes:
- `torch`: tensor computation, neural networks, automatic differentiation.
- `transformers`: Hugging Face models and tokenizers, including ESM2.
- `peft`: parameter-efficient fine-tuning utilities such as LoRA.
- `sklearn`: data splitting, baselines, and metrics.
- `umap-learn`: nonlinear dimensionality reduction for visualization.
- `Bio.SeqUtils.ProtParamData.kd`: Kyte-Doolittle hydrophobicity scale values per amino acid.


In [ ]:
import subprocess, sys

# Install the workshop dependencies once at the start.
packages = [
    "torch",
    "transformers>=4.30",
    "peft>=0.4",
    "scipy",
    "matplotlib",
    "seaborn",
    "tqdm",
    "biopython",
    "scikit-learn",
    "umap-learn",
    "torchao>=0.16.0"
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    print(f"Installed {pkg}")


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import EsmModel, EsmTokenizer
from peft import get_peft_model, LoraConfig, TaskType

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import spearmanr
from tqdm import tqdm
import time, random, warnings

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
# Use GPU when available, otherwise fall back to CPU.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

**Q1. Why do we set a seed?**


---
## 2. Background: ESM2 and finetuning <a id='background'></a>

### Why ESM2?

ESM2 is a protein language model from Meta AI trained on large-scale protein sequence data. In this notebook we use the Hugging Face implementation: https://huggingface.co/docs/transformers/model_doc/esm

A protein language model maps amino-acid sequences to contextual residue embeddings. These embeddings often contain biochemical, evolutionary, and structural signal before we train on our downstream labels.

### Why downstream adaptation?
A pretrained model gives general-purpose representations, but a downstream task may require task-specific information. Here the downstream task is peptide aggregation classification.

### Fine-tuning context
Recent benchmarking work by Schmirler/Rost and co-authors compared protein language model fine-tuning strategies across tasks and found that task-specific supervised fine-tuning often improves downstream prediction; they also discuss LoRA/PEFT as a compute-efficient option for larger models: https://www.nature.com/articles/s41467-024-51844-2

### Four strategies in this practical
#### 1. Amino-acid-count baseline
- Uses only amino-acid composition.
- No protein language model is used.
- This checks whether the task can already be solved from simple sequence composition.

#### 2. Frozen ESM2 baseline
- The ESM2 backbone stays fixed.
- We train only a prediction head on top of ESM2 embeddings.
- This is cheap, simple, and a strong baseline.

#### 3. LoRA / PEFT
- Most pretrained weights stay frozen.
- Small trainable low-rank updates are added inside selected modules.
- Documentation: https://huggingface.co/docs/peft/package_reference/lora

#### 4. Full fine-tuning
- All ESM2 parameters are updated.
- This gives the model maximum flexibility.
- It costs the most memory and compute, and it can overfit on small datasets.

### Related aggregation reference
AggBERT predicts hexapeptide amyloidogenesis using a semi-supervised/fine-tuned ProtBERT approach: https://pmc.ncbi.nlm.nih.gov/articles/PMC10777593/


---
## 3. Dataset and task <a id='data'></a>

In this workshop we use peptide sequences labeled for aggregation. The task is **binary classification**: predict whether a peptide belongs to the positive aggregation class.

### Expected files and directory
Place the files in the same directory where this notebook is run. In Google Colab this is usually `/content/`.

- `train_data.csv` → **Training dataset** used to create the training and validation splits.
- `test_data.csv` → **Test dataset** used only for the final evaluation.

The examples below assume CSV files with columns:

- `Sequence`: amino-acid sequence.
- `label`: binary aggregation label.

### Binary label meaning
In this practical, `label = 1` means the peptide is treated as aggregation-positive / amyloidogenic; `label = 0` means aggregation-negative / non-amyloidogenic. This follows the binary prediction framing used in peptide aggregation benchmarks such as AggBERT: https://pmc.ncbi.nlm.nih.gov/articles/PMC10777593/

### Split strategy
We work with:
- **train**: used to optimize model parameters,
- **validation**: used for model selection and early stopping,
- **test**: used only for final evaluation.



In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

DATA_DIR = Path("/content")  # Change this if your files are stored elsewhere.
TRAIN_FILE = DATA_DIR / "train_data.csv"
TEST_FILE  = DATA_DIR / "test_data.csv"

train_df = pd.read_csv(TRAIN_FILE, index_col=0)
test_df  = pd.read_csv(TEST_FILE, index_col=0)

# Split the original training file into train and validation subsets.
# The stratify argument keeps the label proportions more similar across train and validation.
train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    random_state=SEED,
    shuffle=True,
    stratify=train_df["label"],
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

# Sanity check
print("Train:", len(train_df))
print("Val:  ", len(val_df))
print("Test: ", len(test_df))
print("Columns:", list(train_df.columns))
print(train_df.head())


**Q2. Why would we need both a validation and test set?**

**Q3. Why is it important to check class balance?**

**Q4. Stratification on splitting: why is it important?**
Read the `stratify` parameter in `train_test_split`: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html

**Q5. What other choices can you balance or deliberately unbalance when splitting biological sequence data?**

The next plot shows the label counts per split.

A model can look artificially strong or weak if the data split itself is misleading.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))

colors = {0: '#4C72B0', 1: '#DD8452'}

dfs = {
    'train': train_df,
    'val': val_df,
    'test': test_df,
}

splits = list(dfs.keys())
x = np.arange(len(splits))
width = 0.35

counts = {
    split: df['label'].value_counts().to_dict()
    for split, df in dfs.items()
}

for i, label in enumerate([0, 1]):
    ax.bar(
        x + (i - 0.5) * width,
        [counts[split].get(label, 0) for split in splits],
        width=width,
        label=f'label {label}',
        color=colors[label]
    )

ax.set_xticks(x)
ax.set_xticklabels(splits)
ax.set_xlabel('Dataset Split')
ax.set_ylabel('Count')
ax.set_title('Label Distribution per Split')
ax.legend()

plt.tight_layout()
plt.show()

When you inspect it, ask:

**Q6.** Are train, validation, and test of reasonable size?  
**Q7.** Is one class strongly underrepresented?  
**Q8.** Do the splits look comparable, or is one split distributed very differently from the others?  
**Q9.** If the test set has a different class balance than the training set, what metric would you trust most?

---
## 4. Model architecture overview <a id='architecture'></a>

We use ESM2-8M, the smallest ESM2 variant. The model is here: https://huggingface.co/facebook/esm2_t6_8M_UR50D. The smaller model keeps runtime manageable while still showing the full workflow of protein language model adaptation.

At a high level, ESM2 maps a protein sequence to a contextual embedding for every token:

`sequence -> tokenizer -> transformer encoder -> token/residue representations`

The model output `outputs.last_hidden_state` has shape `(B, L, D)`:

- `B` = batch size, the number of sequences processed together.
- `L` = tokenized sequence length after adding special tokens and padding.
- `D` = hidden dimension / embedding dimension, 320 for ESM2-8M.

From those token-level representations, we derive a sequence-level representation for classification.

### Two common pooling choices for sequence-level representations
In this notebook you will see two strategies:

- **Mean pooling over residues:** average all residue embeddings after removing special tokens.
- **CLS-token prediction:** use the special beginning-of-sequence representation, indexed with `hidden[:, 0, :]`, as the global summary.


In [ ]:
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "facebook/esm2_t6_8M_UR50D"

print(f"Loading tokenizer and model: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
esm_base  = AutoModel.from_pretrained(MODEL_NAME)
EMBED_DIM = esm_base.config.hidden_size   # 320 for ESM2-8M

total_params = sum(p.numel() for p in esm_base.parameters())

print(f"Model loaded: {total_params/1e6:.1f}M parameters")
print(f"Embedding dim: {esm_base.config.hidden_size}")
print(f"Num layers:    {esm_base.config.num_hidden_layers}")
print(f"Num heads:     {esm_base.config.num_attention_heads}")

**Q10. Make an architecture figure.**
Sketch the dimensionalities for one batch: raw sequences → token IDs `(B, L)` → hidden states `(B, L, D)` → pooled sequence vectors `(B, D)` → logits `(B,)`.

The next cell wraps sequences and labels into a PyTorch `Dataset`, then builds `DataLoader`s.

This is where three practical issues are handled:

- tokenization into model inputs,
- padding so variable-length proteins can share a batch,
- truncation to a maximum sequence length.

**STUDENT TASK 1: Tokenizer coding task.**
Fill in the `encoded = ...` line inside `collate_fn`.

Use the tokenizer documentation and examples here: https://huggingface.co/docs/transformers/main_classes/tokenizer

Parameters you need to think about:
- `list(seqs)`: the batch of amino-acid sequences.
- `return_tensors="pt"`: return PyTorch tensors.
- `padding=...`: pad shorter sequences in the batch.
- `truncation=...`: cut sequences longer than `max_length`.
- `max_length=...`: maximum number of tokens accepted in this practical.


In [ ]:
from torch.utils.data import Dataset, DataLoader

class ProteinDataset(Dataset):
    """Wraps sequences + labels into a PyTorch Dataset."""
    def __init__(self, sequences, labels):
        self.sequences = list(sequences)
        self.labels    = torch.tensor(list(labels), dtype=torch.float32)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

def collate_fn(batch):
    # Tokenize a whole batch at once so sequences can have different lengths.
    seqs, labels = zip(*batch)

    # ──────────────────────────────────────────────────────────────────────────────
    # STUDENT TASK 1: Tokenizer coding task.
    # ──────────────────────────────────────────────────────────────────────────────
    encoded = None  # TODO: call tokenizer(...)

    return encoded, torch.stack(labels)

train_dataset = ProteinDataset(
    sequences=train_df["Sequence"],
    labels=train_df["label"]
)

val_dataset = ProteinDataset(
    sequences=val_df["Sequence"],
    labels=val_df["label"]
)

test_dataset = ProteinDataset(
    sequences=test_df["Sequence"],
    labels=test_df["label"]
)

# Keep the batch size modest because transformer memory grows quickly with sequence length.
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)


batch = next(iter(train_loader))
inputs, labels = batch

print("Input IDs shape:", inputs["input_ids"].shape)
print("Attention mask shape:", inputs["attention_mask"].shape)
print("Labels shape:", labels.shape)


**Q11.** What happens to memory use when `max_length` increases?  
**Q12.** What information might be lost when truncation is enabled?

---
## 5. Amino-acid-count baseline <a id='aacount-baseline'></a>

Before using ESM2, we test a simple baseline: count amino acids and train a classical classifier. This answers a crucial question: is the dataset already largely predictable from amino-acid composition?

This baseline uses 20 features per sequence: one count for each standard amino acid.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")

def amino_acid_count_features(sequences, normalize=True):
    # Convert sequences to amino-acid count or frequency vectors.
    X = np.zeros((len(sequences), len(AMINO_ACIDS)), dtype=float)
    aa_to_idx = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

    for row, seq in enumerate(sequences):
        for aa in str(seq):
            if aa in aa_to_idx:
                X[row, aa_to_idx[aa]] += 1
        if normalize and X[row].sum() > 0:
            X[row] = X[row] / X[row].sum()
    return X

X_train_counts = amino_acid_count_features(train_df["Sequence"])
X_val_counts   = amino_acid_count_features(val_df["Sequence"])
X_test_counts  = amino_acid_count_features(test_df["Sequence"])

y_train = train_df["label"].values
y_val   = val_df["label"].values
y_test  = test_df["label"].values

# Choose one baseline. Logistic regression is easier to interpret; random forest can model nonlinear interactions.
count_baseline = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)

count_baseline.fit(X_train_counts, y_train)

count_probs = count_baseline.predict_proba(X_test_counts)[:, 1]
count_preds = (count_probs > 0.5).astype(int)

count_results = (
    roc_auc_score(y_test, count_probs),
    accuracy_score(y_test, count_preds),
    f1_score(y_test, count_preds),
)

print("Amino-acid-count baseline:")
print(f"AUC={count_results[0]:.3f} | ACC={count_results[1]:.3f} | F1={count_results[2]:.3f}")


**Q13.** Why is this baseline scientifically useful even if ESM2 performs better?  
**Q14.** If this baseline is almost as strong as ESM2, what does that suggest about the task or dataset?


---
## 6. Inspecting pretrained sequence embeddings with UMAP <a id='umap'></a>

Before doing supervised training, we first inspect the geometry of the pretrained ESM2 embedding space at the sequence level.

### What we extract
In the next cell we:
- pass sequences through the pretrained ESM2 encoder,
- collect token-level embeddings,
- remove special tokens such as BOS/CLS and EOS,
- average the residue embeddings to one vector per sequence,
- calculate the mean Kyte-Doolittle hydrophobicity for each sequence.

`hidden = outputs.last_hidden_state` has shape `(B, L, D)`:
- `B`: batch size,
- `L`: token length including special tokens and padding,
- `D`: embedding dimension.

**STUDENT TASK 2: Mean pooling coding task.**
Fill in the line that computes `avg_embedding` from `h`, where `h` has shape `(residue_length, D)` for one sequence.

PyTorch mean documentation: https://docs.pytorch.org/docs/stable/generated/torch.mean.html

Hydrophobicity note: `kd` is the Kyte-Doolittle scale dictionary from Biopython. The expression `kd.get(aa, np.nan)` returns the hydrophobicity value for amino acid `aa`, or `np.nan` for unknown tokens. See Biopython protein analysis documentation: https://biopython.org/docs/latest/api/Bio.SeqUtils.ProtParam.html



In [ ]:
import torch
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from umap import UMAP
from Bio.SeqUtils.ProtParamData import kd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
esm_base = esm_base.to(device)
esm_base.eval()

def extract_avg_embeddings_and_hydrophobicity(dataloader, model):
    all_avg_embeddings = []
    all_seq_hydrophobicity = []

    model.eval()

    for batch in tqdm(dataloader):
        inputs, labels = batch
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            hidden = outputs.last_hidden_state  # (B, L, D)

        input_ids = inputs["input_ids"]

        # Process each sequence separately because the real sequence length differs inside a padded batch.
        for i in range(hidden.size(0)):
            seq_len = (input_ids[i] != tokenizer.pad_token_id).sum().item()

            # remove CLS + EOS
            h = hidden[i, 1:seq_len-1, :]  # (L, D)
            tokens = input_ids[i, 1:seq_len-1]
            residues = tokenizer.convert_ids_to_tokens(tokens)

            # average embedding per sequence
            # Mean pooling gives one vector per protein sequence.
            # ──────────────────────────────────────────────────────────────────────────────
            # STUDENT TASK 2: Mean pooling coding task
            # ──────────────────────────────────────────────────────────────────────────────
            avg_embedding = None  # TODO: mean-pool h across the residue dimension (dim=0)
            all_avg_embeddings.append(avg_embedding.cpu().numpy())

            # average hydrophobicity per sequence
            hydro_values = [kd.get(aa, np.nan) for aa in residues]
            hydro_values = np.array(hydro_values, dtype=float)

            # ignore NaNs if there are uncommon tokens
            seq_hydro = np.nanmean(hydro_values)
            all_seq_hydrophobicity.append(seq_hydro)

    all_avg_embeddings = np.stack(all_avg_embeddings, axis=0)
    all_seq_hydrophobicity = np.array(all_seq_hydrophobicity, dtype=float)

    return all_avg_embeddings, all_seq_hydrophobicity


embeddings, seq_hydrophobicity = extract_avg_embeddings_and_hydrophobicity(
    train_loader,
    esm_base,
)

print("Embeddings shape:", embeddings.shape)
print("Hydrophobicity shape:", seq_hydrophobicity.shape)


**Q15.** Why do we remove BOS/CLS and EOS before mean pooling?  
**Q16.** What happens to the pooled vector if one sequence is much longer than another?

### UMAP of peptide embeddings colored by hydrophobicity

Each point represents a peptide, obtained by averaging its residue embeddings. The color indicates the average hydrophobicity of that peptide. Hydrophobicity is a known driver of aggregation, so we check whether this feature is reflected in the embedding space.

UMAP documentation: https://umap-learn.readthedocs.io/en/latest/parameters.html

In the next cell, `n_components=2` means we project to two dimensions for plotting. `n_neighbors=10` controls how much local versus broader neighborhood structure UMAP tries to preserve.

Keep in mind that UMAP distorts global structure, so focus on local neighborhoods rather than large-scale distances.


In [ ]:
# UMAP
umap = UMAP(
    n_neighbors=10,
    n_components=2,
    metric="euclidean",
    random_state=42
)

# Reduce the sequence embeddings to 2D for visualization only.
emb_2d = umap.fit_transform(embeddings)

# Plot colored by sequence-average hydrophobicity
plt.figure(figsize=(10, 8))
sc = plt.scatter(
    emb_2d[:, 0],
    emb_2d[:, 1],
    c=seq_hydrophobicity,
    s=10,
    alpha=0.7
)

plt.colorbar(sc, label="Mean Kyte-Doolittle hydrophobicity")
plt.title("UMAP of Average ESM2 Sequence Embeddings Colored by Hydrophobicity")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.tight_layout()
plt.show()



**Q17.** What do you expect to happen if `n_neighbors` is much smaller, for example 5?  
**Q18.** What does `n_components=2` make possible, and what information might it throw away?

---
## 7. Strategy 1: Frozen baseline <a id='frozen'></a>

### Idea
Use the pretrained ESM2 as a fixed embedding extractor:
- Backbone weights are frozen.
- Sequence representations are computed from the frozen encoder.
- Only a lightweight prediction head is trained.

### Intuition
The pretrained model already encodes rich biochemical and structural priors. This strategy assumes those representations are sufficiently general, and that a simple downstream mapping is enough to solve the task.

### Pooling strategy
Mean pooling over residue embeddings, excluding special tokens.

**STUDENT TASK 3: Prediction head forward pass.**
Implement the forward pass in `PredictionHead`. Documentation for `nn.Sequential`: https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html



In [ ]:
from torch import nn

class PredictionHead(nn.Module):
    """Simple MLP prediction head (binary classification)."""
    def __init__(self, embed_dim: int, hidden_dim: int = 32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        # ──────────────────────────────────────────────────────────────────────────────
        # STUDENT TASK 3: Prediction head forward pass
        # ──────────────────────────────────────────────────────────────────────────────
        return None # TODO: pass x through self.net and return logits with shape (B,) using squeeze(-1)


class FrozenESM2Model(nn.Module):
    """
    ESM2 with all weights FROZEN.
    Only the prediction head is trained.
    Pooling: mean over residue embeddings (excluding BOS/EOS via slicing).
    """
    def __init__(self, esm_encoder, embed_dim, head_hidden=32):
        super().__init__()
        self.encoder = esm_encoder
        self.head    = PredictionHead(embed_dim, head_hidden)

        # Freeze the encoder so only the small prediction head learns.
        for param in self.encoder.parameters():
            param.requires_grad = False

    def forward(self, encoded):
        with torch.no_grad():
            outputs = self.encoder(**encoded)

        hidden = outputs.last_hidden_state  # given: shape (B, L, D)
        hidden_no_bos_eos = hidden[:, 1:-1, :]
        pooled = hidden_no_bos_eos.mean(dim=1)

        return self.head(pooled)


# Build model
frozen_model = FrozenESM2Model(
    esm_encoder=AutoModel.from_pretrained(MODEL_NAME),
    embed_dim=EMBED_DIM
).to(DEVICE)



**Q19.** Which parameters still have `requires_grad=True` in the frozen model?  
**Q20.** Why is the encoder wrapped in `torch.no_grad()` here?

The next training cell runs optimization for the frozen baseline. This function is reused for the other ESM2 strategies.

### What the training loop does
- trains on the training split,
- monitors validation loss,
- applies early stopping,
- reports final test metrics.

### Metrics used here
We report ROC AUC, accuracy, and F1. In imbalanced binary classification, AUC and F1 often tell a more useful story than accuracy alone.

**STUDENT TASK 4: Training-loop coding task.**
Fill in the optimizer and only optimize the parameters where p.requires_grad is True

Useful documentation:
- Adam optimizer: https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html


In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

def train_model(
    model,
    train_loader,
    val_loader,
    test_loader,
    n_epochs=10,
    lr=5e-4,
    device=DEVICE,
    patience=5,
):
    # Only optimize parameters that are marked as trainable.
    # ──────────────────────────────────────────────────────────────────────────────
    # STUDENT TASK 4: Training-loop coding task.
    # ──────────────────────────────────────────────────────────────────────────────
    optimizer = None # TODO: create Adam over parameters where p.requires_grad is True

    criterion = nn.BCEWithLogitsLoss()

    best_val_loss = float('inf')
    best_state = None
    epochs_no_improve = 0

    # The same training loop is reused for frozen, LoRA, and full fine-tuning.
    for epoch in range(1, n_epochs + 1):
        # ── Training ─────────────────────────────
        model.train()
        train_loss = 0

        for encoded, labels in train_loader:
            encoded = {k: v.to(device) for k, v in encoded.items()}
            labels  = labels.float().to(device)

            optimizer.zero_grad()
            logits = model(encoded)
            loss   = criterion(logits, labels)  # TODO: compute the loss with criterion(logits, labels)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        # ── Validation ───────────────────────────
        model.eval()
        val_loss = 0

        all_probs = []
        all_labels = []

        with torch.no_grad():
            for encoded, labels in val_loader:
                encoded = {k: v.to(device) for k, v in encoded.items()}
                labels  = labels.float().to(device)

                logits = model(encoded)  # run the forward pass
                loss   = criterion(logits, labels)  # compute the loss with criterion(logits, labels)

                val_loss += loss.item()

                # Convert logits to probabilities for the validation metrics.
                probs = torch.sigmoid(logits)

                all_probs.append(probs.cpu())
                all_labels.append(labels.cpu())

        val_loss /= len(val_loader)

        all_probs = torch.cat(all_probs).numpy()
        all_labels = torch.cat(all_labels).numpy()

        preds = (all_probs > 0.5).astype(int)

        val_acc = accuracy_score(all_labels, preds)
        val_f1  = f1_score(all_labels, preds)
        val_auc = roc_auc_score(all_labels, all_probs)

        print(f"Epoch {epoch:3d} | "
              f"train_loss={train_loss:.3f} | "
              f"val_loss={val_loss:.3f} | "
              f"val_acc={val_acc:.3f} | "
              f"val_f1={val_f1:.3f} | "
              f"val_auc={val_auc:.3f}")

        # ── Early stopping ───────────────────────
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            # Save the best checkpoint according to validation loss.
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    # ── Load best model ─────────────────────────
    if best_state is not None:
        model.load_state_dict(best_state)

    # ── Test evaluation ─────────────────────────
    model.eval()

    all_probs = []
    all_labels = []

    with torch.no_grad():
        for encoded, labels in test_loader:
            encoded = {k: v.to(device) for k, v in encoded.items()}
            labels  = labels.float().to(device)

            logits = model(encoded)  # run the forward pass

            # Convert logits to probabilities for the validation metrics.
            probs = torch.sigmoid(logits)

            all_probs.append(probs.cpu())
            all_labels.append(labels.cpu())

    all_probs = torch.cat(all_probs).numpy()
    all_labels = torch.cat(all_labels).numpy()

    preds = (all_probs > 0.5).astype(int)

    test_acc = accuracy_score(all_labels, preds)
    test_f1  = f1_score(all_labels, preds)
    test_auc = roc_auc_score(all_labels, all_probs)

    print("\nTest performance:")
    print(f"ACC={test_acc:.3f} | F1={test_f1:.3f} | AUC={test_auc:.3f}")

    return test_auc, test_acc, test_f1

frozen_results = train_model(
    model=frozen_model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    n_epochs=10,
    lr=5e-3,
    device=DEVICE,
    patience=5,
)


**Q21.** Why do we use `BCEWithLogitsLoss` rather than applying `sigmoid` first and then using binary cross-entropy?  
**Q22.** Why does the optimizer filter for `p.requires_grad`?  
**Q23.** What does early stopping protect against?

---
## 8. Strategy 2: LoRA fine-tuning <a id='lora'></a>

### Idea
Apply Low-Rank Adaptation (LoRA) to the pretrained ESM2:
- Backbone is mostly frozen.
- Trainable low-rank updates are injected into selected modules.
- Prediction head is trained jointly.

LoRA documentation: https://huggingface.co/docs/peft/package_reference/lora

### Target modules
The `target_modules` parameter tells PEFT where to insert LoRA adapters. For ESM2 attention blocks, common targets include projection layers such as `query`, `key`, `value`, and `dense`. You can inspect module names using:

```python
for name, module in base.named_modules():
    print(name)
```

### Pooling strategy

**STUDENT TASK 5: CLS-token task.**
Use the CLS token representation: `hidden[:, 0, :]`. For more info you can refer to the 'Pooling Techniques' slides from lecture 9


In [ ]:
class LoRAESM2Model(nn.Module):
    """
    ESM2 with LoRA adapters on attention layers.
    Prediction from CLS token.
    """
    def __init__(self, model_name, embed_dim, head_hidden=32, lora_rank=4, lora_alpha=1):
        super().__init__()

        base = EsmModel.from_pretrained(model_name)

        # LoRA inserts small trainable adapters into the attention blocks.
        lora_cfg = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            r=lora_rank,
            lora_alpha=lora_alpha,
            lora_dropout=0.1,
            target_modules=["query", "key", "value", "dense"],
            bias="none",
        )
        self.encoder = get_peft_model(base, lora_cfg)
        self.head = PredictionHead(embed_dim, head_hidden)

    def forward(self, encoded):
        outputs = self.encoder(**encoded)
        hidden = outputs.last_hidden_state

        # CLS token = first token
        # ──────────────────────────────────────────────────────────────────────────────
        # STUDENT TASK 5: CLS-token task
        # ──────────────────────────────────────────────────────────────────────────────
        cls_token = # TODO: Use the CLS token as the sequence representation.

        return self.head(cls_token)


lora_model = LoRAESM2Model(
    model_name=MODEL_NAME,
    embed_dim=EMBED_DIM,
    lora_rank=4,
    lora_alpha=1,
).to(DEVICE)

lora_model.encoder.print_trainable_parameters()


**Q24.** What do `r`, `lora_alpha`, and `lora_dropout` control?  
**Q25.** How can you verify the shape of `hidden` before selecting `hidden[:, 0, :]`?  
**Q26.** Why might LoRA be especially attractive for larger protein language models?

In [ ]:
print("\nTraining LoRA model...")
lora_results = train_model(
    model=lora_model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    n_epochs=10,
    lr=5e-3,
    device=DEVICE,
    patience=5,
)


---
## 9. Strategy 3: Full fine-tuning <a id='full'></a>

### Idea
Fully fine-tune the pretrained ESM2 model:
- All backbone weights are trainable.
- Prediction head is trained jointly.

### Intuition
When the downstream task differs substantially from pretraining, the model may need to reshape its internal representations. Full fine-tuning allows adaptation across the entire parameter space.

### Pooling strategy
The CLS token is used as representation: `hidden[:, 0, :]`.


In [ ]:
class FullFineTuneESM2Model(nn.Module):
    """
    ESM2 fully fine-tuned.
    Per-protein prediction from the CLS token (position 0),
    as recommended by the ESM2 authors and the Schmirler et al. paper.
    """
    def __init__(self, model_name, embed_dim, head_hidden=32):
        super().__init__()
        self.encoder = EsmModel.from_pretrained(model_name)
        self.head    = PredictionHead(embed_dim, head_hidden)
        # All parameters are trainable by default — no freezing

    def forward(self, encoded):
        outputs = self.encoder(**encoded)
        hidden  = outputs.last_hidden_state

        # Full fine-tuning keeps the same CLS-based readout, but now the whole encoder can adapt.
        pooled = hidden[:, 0, :]

        return self.head(pooled)


full_model = FullFineTuneESM2Model(
    model_name=MODEL_NAME,
    embed_dim=EMBED_DIM
).to(DEVICE)



**Q27.** Why might full fine-tuning overfit more easily than LoRA on a small dataset?  
**Q28.** What extra compute or memory costs do you expect compared with the frozen baseline?

In [ ]:
print("\nTraining full model...")
full_results = train_model(
    model=full_model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    n_epochs=10,
    lr=5e-4,
    device=DEVICE,
    patience=5,
)

---
## 10. Comparison and interpretation <a id='comparison'></a>

At this point we have four regimes:
- amino-acid-count baseline,
- frozen ESM2 backbone,
- LoRA / PEFT,
- full fine-tuning.

Now we compare them in a single place.

**STUDENT TASK 6: Parameter-count task.**
Complete the function that counts trainable and total parameters. Hint: each parameter tensor has `.numel()`.

`requires_grad` documentation: https://docs.pytorch.org/docs/stable/generated/torch.Tensor.requires_grad.html


In [ ]:
def count_trainable_params(model):
    # ──────────────────────────────────────────────────────────────────────────────
    # STUDENT TASK 6: Parameter-count task
    # ──────────────────────────────────────────────────────────────────────────────
    trainable = # TODO: sum p.numel() for trainable parameters
    total     = # TODO: sum p.numel() for all parameters
    return trainable, total

# Collect the final metrics in one place so the strategies are easy to compare.
results = {
    'AA counts\n(logistic)': {
        'test_auc': count_results[0],
        'test_acc': count_results[1],
        'test_f1':  count_results[2],
        'model':    None,
        'pooling':  'composition',
        'color':    '#8172B2',
        'trainable_pct': 0.0,
    },
    'Frozen\n(static embeddings)': {
        'test_auc': frozen_results[0],
        'test_acc': frozen_results[1],
        'test_f1':  frozen_results[2],
        'model':    frozen_model,
        'pooling':  'mean',
        'color':    '#4C72B0',
    },
    'LoRA\n(PEFT)': {
        'test_auc': lora_results[0],
        'test_acc': lora_results[1],
        'test_f1':  lora_results[2],
        'model':    lora_model,
        'pooling':  'cls',
        'color':    '#55A868',
    },
    'Full Fine-tune\n(CLS token)': {
        'test_auc': full_results[0],
        'test_acc': full_results[1],
        'test_f1':  full_results[2],
        'model':    full_model,
        'pooling':  'cls',
        'color':    '#DD8452',
    },
}

print("\n" + "=" * 80)
print(f"{'Strategy':<30} {'AUC':>8} {'Acc':>8} {'F1':>8} {'Trainable%':>12}")
print("-" * 80)

for name, res in results.items():
    label = name.replace('\n', ' ')

    if res['model'] is None:
        pct = res['trainable_pct']
    else:
        t, total = count_trainable_params(res['model'])
        pct = 100 * t / total

    print(f"{label:<30} "
          f"{res['test_auc']:>8.4f} "
          f"{res['test_acc']:>8.4f} "
          f"{res['test_f1']:>8.4f} "
          f"{pct:>11.2f}%")

print("=" * 80)


**Q29.** Which method has the best AUC?  
**Q30.** Which method has the best performance per trainable parameter?  
**Q31.** Are the observed differences statistically significant to trust from one split and one seed?

The following plot places each strategy on an efficiency axis.

### How to read it
- The x-axis shows the percentage of trainable parameters.
- The y-axis shows downstream test F1.

A point is attractive when it lies high on the y-axis while staying far left on the x-axis: strong performance with minimal trainable capacity.

**Important interpretation note.**
The final figure is descriptive, not a significance test. Differences between methods may not be statistically significant from one random seed and one train/test split.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

for name, res in results.items():
    label = name.replace('\n', ' ')

    if res['model'] is None:
        pct = res['trainable_pct']
    else:
        t, total = count_trainable_params(res['model'])
        pct = 100 * t / total

    f1 = res['test_auc']

    ax.scatter(
        pct,
        f1,
        s=100,
        color=res['color'],
        edgecolors='black',
        linewidth=1.5,
        alpha=0.85,
    )

    ax.annotate(
        f'{label}\n(AUC={f1:.3f})',
        (pct, f1),
        textcoords='offset points',
        xytext=(10, 0),
        fontsize=9,
        va='center',
    )

ax.set_xlabel('Trainable Parameters (%)')
ax.set_ylabel('Test AUC')
ax.set_title('Efficiency Plot: descriptive only, no error bars')
ax.set_xscale('symlog', linthresh=0.1)

plt.tight_layout()
plt.savefig('efficiency_plot.png', dpi=150)
plt.show()



**Q32.** How could you estimate uncertainty here? Consider repeated seeds, bootstrapped confidence intervals, cross-validation, or error bars over multiple train/validation/test splits.

**Q33.** How would you compare your findings to the Schmirler/Rost fine-tuning paper? Do your results support the same conclusion for this peptide aggregation task?

---
## 11. Takeaways and exercises <a id='exercises'></a>

### Key takeaways
- A simple amino-acid-count baseline is essential; it tells you how much signal is already present in composition.
- Pretrained protein language models can encode meaningful biochemical structure before task-specific training.
- A frozen baseline is a real baseline, not a formality.
- LoRA can preserve most pretrained weights while adapting efficiently.
- Full fine-tuning offers maximum flexibility but has the highest computational cost and overfitting risk.
- One split and one seed are not enough to claim that small differences are significant.

### Additional exercises
1. Replace mean pooling with CLS-token prediction in the frozen model and compare results.
2. Run UMAP on embeddings after fine-tuning and compare with the pretrained UMAP.
3. Combine architecture and dimensionality analysis: make a figure showing each model route from sequence input to logits, including tensor shapes and trainable/frozen components.
